# 🎨 Google Photos AI 超能工坊 (Private AI Photo Studio)

本筆記本直接銜接 `Organized_Photos/` 整理相簿，**100% 在 Google 雲端 GPU 執行**，不消耗本機電腦效能！

### 核心三大超能力：
1. **👤 人臉識別與自動人物分群**：媲美 Google 相簿，InsightFace 深度特徵 + DBSCAN 聚類，自動歸納核心人物。
2. **🔍 繁體中文語義搜圖**：Multilingual CLIP 向量檢索，不用標籤，打字「看海的背影」、「生日吹蠟燭」0.05 秒精準出圖。
3. **🌐 雙軌 HTML 體驗**：內建 Gradio 現代 HTML5 操作面板（支援手機連線），並可一鍵匯出單檔 `Family_Album.html` 離線相簿！

> **💡 建議**：請先至 Colab 選單「執行階段」➜「變更執行階段類型」，確認硬體加速器已選擇 **GPU (T4 / L4 / A100)**。

## 步驟 1：掛載 Google 雲端硬碟 (Google Drive)

In [1]:
from google.colab import drive
import os, sys, shutil, glob, json, hashlib, base64, io, subprocess, time
from pathlib import Path

def ensure_drive_connected():
    # 確保 /content/drive 連線正常，若遇 Errno 107 瞬斷自動重新掛載
    try:
        _ = os.listdir('/content/drive/MyDrive')
    except OSError:
        print('\n🔄 偵測到 Google Drive 通道瞬斷，正在自動重新連線...')
        subprocess.run(['fusermount', '-uz', '/content/drive'], stderr=subprocess.DEVNULL, timeout=15)
        subprocess.run(['umount', '-l', '/content/drive'], stderr=subprocess.DEVNULL, timeout=15)
        drive.mount('/content/drive', force_remount=True)
        print('✓ Google Drive 自動重新連線成功！')

try:
    _ = os.listdir('/content/drive/MyDrive')
except OSError:
    subprocess.run(['fusermount', '-uz', '/content/drive'], stderr=subprocess.DEVNULL, timeout=15)
    subprocess.run(['umount', '-l', '/content/drive'], stderr=subprocess.DEVNULL, timeout=15)

drive.mount('/content/drive', force_remount=True)
print('✓ Google Drive 掛載成功！')

Mounted at /content/drive
✓ Google Drive 掛載成功！


## 步驟 2：安裝核心 AI 與現代 HTML5 介面套件

In [2]:
# 安裝 Gradio 現代 Web 面板、Sentence-Transformers (含多模態影像支援)、InsightFace 與圖像解碼套件
!pip install -q gradio "sentence-transformers[image]" insightface "pillow>=10.0,<13.0" pillow-heif scikit-learn
!pip install -q onnxruntime-gpu
!pip uninstall -y -q onnxruntime >/dev/null 2>&1 || true
!pip install -q --force-reinstall --no-deps onnxruntime-gpu
import pillow_heif
pillow_heif.register_heif_opener()
print('✓ 核心 AI 函式庫安裝完成（已啟用多模態影像支援、GPU ONNX 與 iPhone HEIC/HEIF 解碼）！')

✓ 核心 AI 函式庫安裝完成（已啟用多模態影像支援、GPU ONNX 與 iPhone HEIC/HEIF 解碼）！


## 步驟 3：目錄設定與相片掃描（自動排除 Duplicates 重複區）

In [3]:
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None

DRIVE_ROOT = Path('/content/drive/MyDrive')
PHOTOS_DIR = DRIVE_ROOT / 'Organized_Photos'
ARCHIVE_DIR = PHOTOS_DIR / 'Photos_Archive' if (PHOTOS_DIR / 'Photos_Archive').exists() else PHOTOS_DIR
INDEX_DIR = DRIVE_ROOT / 'Photo_AI_Index'
RESTORED_DIR = DRIVE_ROOT / 'Restored_Photos'

INDEX_DIR.mkdir(parents=True, exist_ok=True)
RESTORED_DIR.mkdir(parents=True, exist_ok=True)

# 若已有新歸檔相片想重算全相簿人臉群組，將下方改為 True
FORCE_RECLUSTER_FACES = False

# 支援的圖片格式（排除 Duplicates 重複隔離區、Restored 修復區與隱藏暫存檔）
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.heic', '.heif'}
all_photos = [
    p for p in ARCHIVE_DIR.rglob('*')
    if p.is_file()
    and p.suffix.lower() in VALID_EXTS
    and not p.name.startswith('.')
    and 'Duplicates' not in p.parts
    and 'Restored' not in p.parts
]

print(f'相簿來源目錄：{ARCHIVE_DIR}')
print(f'已載入有效相片總數：{len(all_photos)} 張')
if len(all_photos) == 0:
    print('⚠️ 目前 Organized_Photos 目錄下尚無相片，請先確認第一本相簿整理筆記本已完成。')

相簿來源目錄：/content/drive/MyDrive/Organized_Photos/Photos_Archive
已載入有效相片總數：17227 張


## 步驟 4：建立「繁體中文語義搜圖」向量索引
使用 Multilingual-CLIP 多模態模型，將每張照片編碼為 512 維語義特徵向量。
（支援增量更新：既有照片秒級載入，僅針對新增照片計算特徵）

In [4]:
import json, os, subprocess, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import torch
import numpy as np
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None
import pillow_heif
pillow_heif.register_heif_opener()
from sentence_transformers import SentenceTransformer

if 'ensure_drive_connected' not in globals():
    def ensure_drive_connected():
        try:
            _ = os.listdir('/content/drive/MyDrive')
        except OSError:
            from google.colab import drive
            subprocess.run(['fusermount', '-uz', '/content/drive'], stderr=subprocess.DEVNULL, timeout=15)
            subprocess.run(['umount', '-l', '/content/drive'], stderr=subprocess.DEVNULL, timeout=15)
            drive.mount('/content/drive', force_remount=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'目前使用運算裝置：{device.upper()}')

# clip-ViT-B-32-multilingual-v1 僅支援多語系「文字」輸入；影像編碼須搭配對齊同一 512 維空間的 clip-ViT-B-32
text_model = SentenceTransformer('clip-ViT-B-32-multilingual-v1', device=device)

clip_index_file = INDEX_DIR / 'clip_embeddings.npy'
clip_paths_file = INDEX_DIR / 'clip_paths.json'

def _save_clip_cache(paths_list, emb_blocks):
    if not emb_blocks:
        return None
    stacked = np.vstack(emb_blocks).astype(np.float32)
    payload_json = json.dumps([str(p) for p in paths_list], ensure_ascii=False)
    for _attempt in range(2):
        try:
            ensure_drive_connected()
            tmp_npy = INDEX_DIR / '.clip_embeddings.tmp.npy'
            tmp_json = INDEX_DIR / '.clip_paths.tmp.json'
            with open(tmp_npy, 'wb') as f:
                np.save(f, stacked)
            tmp_json.write_text(payload_json, encoding='utf-8')
            tmp_npy.replace(clip_index_file)
            tmp_json.replace(clip_paths_file)
            return stacked
        except OSError:
            if _attempt == 1:
                raise
            time.sleep(1.0)
    return stacked

cached_map = {}
if clip_index_file.exists() and clip_paths_file.exists():
    try:
        ensure_drive_connected()
        old_embs = np.load(clip_index_file)
        old_paths = json.loads(clip_paths_file.read_text(encoding='utf-8'))
        if len(old_embs) == len(old_paths) and len(old_paths) > 0:
            cached_map = {p: emb for p, emb in zip(old_paths, old_embs)}
            print(f'✓ 已讀取既有語義快取：{len(cached_map)} 筆')
    except Exception as e:
        print(f'快取重設：{e}')

photo_paths = []
embeddings_list = []
new_photos = []

for p in all_photos:
    p_str = str(p)
    if p_str in cached_map:
        photo_paths.append(p)
        embeddings_list.append(cached_map[p_str].reshape(1, -1))
    else:
        new_photos.append(p)

def _load_clip_thumb(p: Path):
    try:
        with Image.open(p) as im:
            im_fixed = ImageOps.exif_transpose(im).convert('RGB')
            im_fixed.thumbnail((512, 512))
            im_fixed.load()
            return p, im_fixed
    except OSError as exc:
        if getattr(exc, 'errno', None) == 107 or 'Transport endpoint is not connected' in str(exc):
            raise
        return p, None
    except Exception:
        return p, None

failed_clip = 0
if new_photos:
    print(f'載入視覺向量模型 clip-ViT-B-32，開始計算新增相片特徵：共 {len(new_photos)} 張...')
    img_model = SentenceTransformer('clip-ViT-B-32', device=device)
    batch_size = 64
    since_last_save = 0
    for i in range(0, len(new_photos), batch_size):
        batch_paths = new_photos[i:i+batch_size]
        decoded = None
        for _attempt in range(2):
            try:
                ensure_drive_connected()
                with ThreadPoolExecutor(max_workers=8) as pool:
                    decoded = list(pool.map(_load_clip_thumb, batch_paths))
                break
            except OSError:
                if _attempt == 1:
                    raise
                time.sleep(1.0)
        batch_images = []
        valid_batch_paths = []
        for p_item, im_obj in (decoded or []):
            if im_obj is not None:
                batch_images.append(im_obj)
                valid_batch_paths.append(p_item)
            else:
                failed_clip += 1
        if batch_images:
            with torch.no_grad():
                embs = np.atleast_2d(
                    img_model.encode(batch_images, batch_size=len(batch_images), convert_to_numpy=True, normalize_embeddings=True)
                ).astype(np.float32)
            embeddings_list.append(embs)
            photo_paths.extend(valid_batch_paths)
            since_last_save += len(valid_batch_paths)
            del batch_images
        done_now = min(i + batch_size, len(new_photos))
        if since_last_save >= 1024:
            stacked_ckpt = _save_clip_cache(photo_paths, embeddings_list)
            if stacked_ckpt is not None:
                embeddings_list = [stacked_ckpt]
            since_last_save = 0
            print(f'進度：{done_now}/{len(new_photos)} 張（已寫入檢查點快取）...')
        elif (i // batch_size) % 10 == 0 or done_now == len(new_photos):
            print(f'進度：{done_now}/{len(new_photos)} 張...')
    del img_model
    if device == 'cuda':
        torch.cuda.empty_cache()

if embeddings_list:
    photo_embeddings = _save_clip_cache(photo_paths, embeddings_list) if new_photos or not clip_index_file.exists() else np.vstack(embeddings_list).astype(np.float32)
    print(f'✓ 語義索引就緒！共計 {len(photo_paths)} 張' + (f'（跳過損毀或無法解碼 {failed_clip} 張）' if failed_clip else '。'))
else:
    photo_embeddings = np.zeros((0, 512), dtype=np.float32)
    print('⚠️ 無有效相片可建立索引。')

目前使用運算裝置：CUDA


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

載入視覺向量模型 clip-ViT-B-32，開始計算新增相片特徵：共 17227 張...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

進度：64/17227 張...
進度：704/17227 張...
進度：1024/17227 張（已寫入檢查點快取）...
進度：1344/17227 張...
進度：1984/17227 張...
進度：2048/17227 張（已寫入檢查點快取）...
進度：2624/17227 張...
進度：3072/17227 張（已寫入檢查點快取）...
進度：3264/17227 張...
進度：3904/17227 張...
進度：4096/17227 張（已寫入檢查點快取）...
進度：4544/17227 張...
進度：5120/17227 張（已寫入檢查點快取）...
進度：5184/17227 張...
進度：5824/17227 張...
進度：6144/17227 張（已寫入檢查點快取）...
進度：6464/17227 張...
進度：7104/17227 張...
進度：7168/17227 張（已寫入檢查點快取）...
進度：7744/17227 張...
進度：8192/17227 張（已寫入檢查點快取）...
進度：8384/17227 張...
進度：9024/17227 張...
進度：9216/17227 張（已寫入檢查點快取）...
進度：9664/17227 張...
進度：10240/17227 張（已寫入檢查點快取）...
進度：10304/17227 張...
進度：10944/17227 張...
進度：11264/17227 張（已寫入檢查點快取）...
進度：11584/17227 張...
進度：12224/17227 張...
進度：12288/17227 張（已寫入檢查點快取）...
進度：12864/17227 張...
進度：13312/17227 張（已寫入檢查點快取）...
進度：13504/17227 張...
進度：14144/17227 張...
進度：14336/17227 張（已寫入檢查點快取）...
進度：14784/17227 張...
進度：15424/17227 張（已寫入檢查點快取）...
進度：16064/17227 張...
進度：16448/17227 張（已寫入檢查點快取）...
進度：16704/17227 張...
進度：17227/17227 張...
✓ 語義索引就緒

## 步驟 5：建立「人臉辨識與自動人物分群」索引
使用 InsightFace 辨識人臉，並透過 DBSCAN 聚類自動歸納核心人物群組。

In [5]:
import json, os, subprocess, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
from PIL import Image, ImageOps
from sklearn.cluster import DBSCAN
import onnxruntime as ort
import insightface
from insightface.app import FaceAnalysis

if 'ensure_drive_connected' not in globals():
    def ensure_drive_connected():
        try:
            _ = os.listdir('/content/drive/MyDrive')
        except OSError:
            from google.colab import drive
            subprocess.run(['fusermount', '-uz', '/content/drive'], stderr=subprocess.DEVNULL, timeout=15)
            subprocess.run(['umount', '-l', '/content/drive'], stderr=subprocess.DEVNULL, timeout=15)
            drive.mount('/content/drive', force_remount=True)

faces_cluster_file = INDEX_DIR / 'people_clusters.json'
faces_vec_cache_file = INDEX_DIR / 'faces_raw_vectors.npy'
faces_meta_cache_file = INDEX_DIR / 'faces_raw_meta.json'

def _save_face_cache(scanned_set, f_recs, f_vecs):
    meta_json = json.dumps({'scanned_paths': sorted(scanned_set), 'face_records': f_recs}, ensure_ascii=False)
    vec_arr = np.asarray(f_vecs, dtype=np.float32) if f_vecs else np.zeros((0, 512), dtype=np.float32)
    for _attempt in range(2):
        try:
            ensure_drive_connected()
            tmp_npy = INDEX_DIR / '.faces_raw_vectors.tmp.npy'
            tmp_json = INDEX_DIR / '.faces_raw_meta.tmp.json'
            with open(tmp_npy, 'wb') as f:
                np.save(f, vec_arr)
            tmp_json.write_text(meta_json, encoding='utf-8')
            tmp_npy.replace(faces_vec_cache_file)
            tmp_json.replace(faces_meta_cache_file)
            return
        except OSError:
            if _attempt == 1:
                raise
            time.sleep(1.0)

if faces_cluster_file.exists() and not globals().get('FORCE_RECLUSTER_FACES', False):
    print('✓ 發現既有人物分群檔案，直接載入中（若想納入新整理相片，請將步驟 3 的 FORCE_RECLUSTER_FACES 設為 True）...')
    ensure_drive_connected()
    with open(faces_cluster_file, 'r', encoding='utf-8') as f:
        people_clusters = json.load(f)
    print(f'✓ 已載入 {len(people_clusters)} 組人物相簿！')
else:
    current_photo_set = {str(p) for p in all_photos}
    scanned_paths = set()
    face_records = []
    face_vectors = []

    if faces_vec_cache_file.exists() and faces_meta_cache_file.exists():
        try:
            ensure_drive_connected()
            meta = json.loads(faces_meta_cache_file.read_text(encoding='utf-8'))
            vecs = np.load(faces_vec_cache_file)
            old_records = meta.get('face_records', [])
            if len(vecs) == len(old_records):
                scanned_paths = {p for p in meta.get('scanned_paths', []) if p in current_photo_set}
                for rec, vec in zip(old_records, vecs):
                    if rec.get('path') in current_photo_set:
                        face_records.append(rec)
                        face_vectors.append(vec)
                print(f'✓ 已載入既有相片人臉特徵快取：已掃描 {len(scanned_paths)} 張相片 / {len(face_vectors)} 張人臉')
        except Exception as e:
            print(f'人臉快取重設：{e}')
            scanned_paths, face_records, face_vectors = set(), [], []

    pending_photos = [p for p in all_photos if str(p) not in scanned_paths]
    if pending_photos:
        avail_providers = ort.get_available_providers()
        selected_providers = [p for p in ('CUDAExecutionProvider', 'CPUExecutionProvider') if p in avail_providers] or ['CPUExecutionProvider']
        use_cuda = 'CUDAExecutionProvider' in selected_providers and torch.cuda.is_available()
        print(f'初始化 InsightFace 人臉辨識引擎 (buffalo_l，ONNX Providers: {avail_providers}，使用: {selected_providers})...')
        face_app = FaceAnalysis(name='buffalo_l', providers=selected_providers)
        face_app.prepare(ctx_id=0 if use_cuda else -1, det_size=(640, 640))

        def _load_face_rgb(p: Path):
            try:
                with Image.open(p) as im:
                    img = ImageOps.exif_transpose(im).convert('RGB')
                    img.thumbnail((1600, 1600))
                    img.load()
                    return p, np.array(img)[:, :, ::-1]
            except OSError as exc:
                if getattr(exc, 'errno', None) == 107 or 'Transport endpoint is not connected' in str(exc):
                    raise
                return p, None
            except Exception:
                return p, None

        print(f'開始掃描新增相片人臉：共 {len(pending_photos)} 張...')
        chunk_size = 64
        processed_new = 0
        since_last_face_save = 0
        for start_idx in range(0, len(pending_photos), chunk_size):
            chunk = pending_photos[start_idx:start_idx + chunk_size]
            decoded_chunk = None
            for _attempt in range(2):
                try:
                    ensure_drive_connected()
                    with ThreadPoolExecutor(max_workers=8) as pool:
                        decoded_chunk = list(pool.map(_load_face_rgb, chunk))
                    break
                except OSError:
                    if _attempt == 1:
                        raise
                    time.sleep(1.0)
            for p_obj, img_bgr in (decoded_chunk or []):
                processed_new += 1
                if img_bgr is None:
                    continue
                scanned_paths.add(str(p_obj))
                try:
                    faces = face_app.get(img_bgr)
                except Exception:
                    continue
                for f in faces:
                    norm_val = float(np.linalg.norm(f.embedding))
                    if norm_val > 0:
                        face_vectors.append((f.embedding / norm_val).astype(np.float32))
                        face_records.append({'path': str(p_obj), 'bbox': [float(x) for x in f.bbox]})
            since_last_face_save += len(chunk)
            if since_last_face_save >= 512 or processed_new == len(pending_photos):
                _save_face_cache(scanned_paths, face_records, face_vectors)
                since_last_face_save = 0
                print(f'人臉掃描進度：{processed_new}/{len(pending_photos)}（累計偵測 {len(face_vectors)} 張人臉，已存檢查點）...')

    print(f'✓ 人臉掃描完成，累計偵測到 {len(face_vectors)} 張人臉！進行 DBSCAN 密度聚類...')
    if len(face_vectors) > 0:
        clustering = DBSCAN(eps=0.55, min_samples=3, metric='cosine')
        labels = clustering.fit_predict(np.vstack(face_vectors))

        people_clusters = {}
        for label, rec in zip(labels, face_records):
            if label == -1:
                continue
            key = f'人物_{label+1:02d}'
            if key not in people_clusters:
                people_clusters[key] = []
            if rec['path'] not in people_clusters[key]:
                people_clusters[key].append(rec['path'])

        clusters_json = json.dumps(people_clusters, ensure_ascii=False, indent=2)
        for _attempt in range(2):
            try:
                ensure_drive_connected()
                tmp_cluster = INDEX_DIR / '.people_clusters.tmp.json'
                tmp_cluster.write_text(clusters_json, encoding='utf-8')
                tmp_cluster.replace(faces_cluster_file)
                break
            except OSError:
                if _attempt == 1:
                    raise
                time.sleep(1.0)
        print(f'🎉 人物分群成功！共聚類出 {len(people_clusters)} 位主要人物。')
    else:
        people_clusters = {}

初始化 InsightFace 人臉辨識引擎 (buffalo_l，ONNX Providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']，使用: ['CUDAExecutionProvider', 'CPUExecutionProvider'])...
download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:03<00:00, 85969.74KB/s]


開始掃描新增相片人臉：共 17227 張...
人臉掃描進度：512/17227（累計偵測 1153 張人臉，已存檢查點）...
人臉掃描進度：1024/17227（累計偵測 1516 張人臉，已存檢查點）...
人臉掃描進度：1536/17227（累計偵測 2296 張人臉，已存檢查點）...
人臉掃描進度：2048/17227（累計偵測 3374 張人臉，已存檢查點）...
人臉掃描進度：2560/17227（累計偵測 4367 張人臉，已存檢查點）...
人臉掃描進度：3072/17227（累計偵測 5259 張人臉，已存檢查點）...
人臉掃描進度：3584/17227（累計偵測 6484 張人臉，已存檢查點）...
人臉掃描進度：4096/17227（累計偵測 7806 張人臉，已存檢查點）...
人臉掃描進度：4608/17227（累計偵測 8662 張人臉，已存檢查點）...
人臉掃描進度：5120/17227（累計偵測 9408 張人臉，已存檢查點）...
人臉掃描進度：5632/17227（累計偵測 10307 張人臉，已存檢查點）...
人臉掃描進度：6144/17227（累計偵測 11537 張人臉，已存檢查點）...
人臉掃描進度：6656/17227（累計偵測 12355 張人臉，已存檢查點）...
人臉掃描進度：7168/17227（累計偵測 13137 張人臉，已存檢查點）...
人臉掃描進度：7680/17227（累計偵測 13873 張人臉，已存檢查點）...
人臉掃描進度：8192/17227（累計偵測 14515 張人臉，已存檢查點）...
人臉掃描進度：8704/17227（累計偵測 15193 張人臉，已存檢查點）...
人臉掃描進度：9216/17227（累計偵測 16097 張人臉，已存檢查點）...
人臉掃描進度：9728/17227（累計偵測 17108 張人臉，已存檢查點）...
人臉掃描進度：10240/17227（累計偵測 17840 張人臉，已存檢查點）...
人臉掃描進度：10752/17227（累計偵測 18460 張人臉，已存檢查點）...
人臉掃描進度：11264/17227（累計偵測 19147 張人臉，已存檢查點）...
人臉掃描進度：11776/17227（累計偵測 20060 張人臉，已存檢查

## 步驟 6：啟動 Gradio 現代 HTML5 互動相簿工作台
點擊下方產生的網址，即可在瀏覽器（桌面與手機均可）享受秒級 AI 搜圖與人物相簿！

In [6]:
import inspect
from pathlib import Path
import numpy as np
import torch
from PIL import Image, ImageOps
import gradio as gr
from sentence_transformers import SentenceTransformer

def _to_gallery_item(p: Path):
    # 瀏覽器不原生支援 HEIC/HEIF，遇蘋果格式時即時轉為預覽縮圖，其餘直接回傳檔案路徑
    if p.suffix.lower() in ('.heic', '.heif'):
        with Image.open(p) as im:
            rgb = ImageOps.exif_transpose(im).convert('RGB')
            rgb.thumbnail((1280, 1280))
            return rgb
    return str(p)

def search_photos(query_text, top_k=18):
    if (
        not query_text
        or not query_text.strip()
        or 'photo_paths' not in globals()
        or 'photo_embeddings' not in globals()
        or len(photo_paths) == 0
        or len(photo_embeddings) == 0
    ):
        return []
    if 'ensure_drive_connected' in globals():
        ensure_drive_connected()
    global text_model
    if 'text_model' not in globals() or text_model is None:
        _dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        text_model = SentenceTransformer('clip-ViT-B-32-multilingual-v1', device=_dev)
    with torch.no_grad():
        q_emb = text_model.encode([query_text], convert_to_numpy=True, normalize_embeddings=True)[0]
    scores = np.dot(photo_embeddings, q_emb)
    top_indices = np.argsort(scores)[::-1][:int(top_k) * 2]
    results = []
    for idx in top_indices:
        p = photo_paths[idx]
        try:
            if not p.exists():
                continue
            score = scores[idx]
            caption = f"{p.name} (相似度: {score:.2f})"
            results.append((_to_gallery_item(p), caption))
        except Exception:
            continue
        if len(results) >= int(top_k):
            break
    return results

def filter_by_person(person_name):
    if not person_name or 'people_clusters' not in globals() or person_name not in people_clusters:
        return []
    if 'ensure_drive_connected' in globals():
        ensure_drive_connected()
    photos = people_clusters[person_name]
    results = []
    for p_str in photos[:30]:
        p = Path(p_str)
        try:
            if not p.exists():
                continue
            results.append((_to_gallery_item(p), p.name))
        except Exception:
            continue
    return results

theme = gr.themes.Soft(primary_hue="teal", neutral_hue="slate")
blocks_kwargs = {"title": "AI 私有相簿超能工坊"}
launch_kwargs = {"share": True, "debug": False}
if "theme" in inspect.signature(gr.Blocks.__init__).parameters:
    blocks_kwargs["theme"] = theme
elif "theme" in inspect.signature(gr.Blocks.launch).parameters:
    launch_kwargs["theme"] = theme

with gr.Blocks(**blocks_kwargs) as demo:
    gr.Markdown("# 📷 Google Photos AI 私有相簿超能工坊")
    gr.Markdown("直接連結 Google One 2TB 相簿，以開源 AI 模型進行語義搜圖與人物瀏覽。")

    with gr.Tabs():
        with gr.TabItem("🔍 繁體中文自然語言搜圖"):
            with gr.Row():
                query_input = gr.Textbox(placeholder="輸入自然語言，如：在海邊看夕陽、穿紅色洋裝、生日吹蠟燭、夜市吃小吃...", label="搜尋描述", scale=4)
                k_slider = gr.Slider(minimum=6, maximum=36, value=18, step=6, label="顯示張數", scale=1)
                search_btn = gr.Button("立即搜尋 ⚡", variant="primary", scale=1)

            search_gallery = gr.Gallery(label="搜尋結果 (點擊可放大檢視)", columns=3, height=600)
            search_btn.click(fn=search_photos, inputs=[query_input, k_slider], outputs=search_gallery)
            query_input.submit(fn=search_photos, inputs=[query_input, k_slider], outputs=search_gallery)

        with gr.TabItem("👤 人物智慧相簿"):
            person_keys = list(people_clusters.keys()) if 'people_clusters' in globals() and isinstance(people_clusters, dict) else []
            with gr.Row():
                person_dropdown = gr.Dropdown(choices=person_keys, value=person_keys[0] if person_keys else None, label="選擇人物", scale=3)
                filter_btn = gr.Button("查看相片 📁", variant="primary", scale=1)

            person_gallery = gr.Gallery(label="該人物相片集", columns=3, height=600)
            filter_btn.click(fn=filter_by_person, inputs=person_dropdown, outputs=person_gallery)
            person_dropdown.change(fn=filter_by_person, inputs=person_dropdown, outputs=person_gallery)

    gr.Markdown("---\n*所有運算 100% 於 Google Colab 私有環境執行，相片資料零外洩。*")

demo.launch(**launch_kwargs)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://86296fb6be06332d39.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 步驟 7：一鍵匯出單檔 `Family_Album.html` (內嵌縮圖免開 Colab 離線網頁)
將人物分群與精華相片縮圖直接內嵌匯出為單一純 HTML 網頁，日後用電腦或手機雙擊即可秒開瀏覽！

In [7]:
import html as html_lib

album_html_file = DRIVE_ROOT / 'Family_Album.html'

html_content = '''<!DOCTYPE html>
<html lang="zh-TW">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>家族回憶錄 - 私有智慧相簿</title>
  <style>
    body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; background: #f8fafc; color: #1e293b; margin: 0; padding: 24px; }
    h1 { font-size: 24px; margin-bottom: 8px; color: #0f766e; }
    .person-card { background: white; border-radius: 12px; padding: 16px; margin-bottom: 24px; box-shadow: 0 2px 8px rgba(0,0,0,0.05); }
    .person-title { font-size: 18px; font-weight: bold; margin-bottom: 12px; color: #0f172a; }
    .photo-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(180px, 1fr)); gap: 12px; }
    .photo-item { border-radius: 8px; overflow: hidden; background: #e2e8f0; display: flex; flex-direction: column; font-size: 12px; color: #475569; text-align: center; }
    .photo-item img { width: 100%; aspect-ratio: 1; object-fit: cover; display: block; }
    .photo-caption { padding: 6px 8px; word-break: break-all; background: #f1f5f9; }
  </style>
</head>
<body>
  <h1>📷 我的私有家庭相簿畫廊</h1>
  <p>由 Colab InsightFace 自動完成人臉聚類，內嵌精華預覽縮圖。</p>
'''

for person, p_list in people_clusters.items():
    safe_person = html_lib.escape(str(person))
    html_content += f'<div class="person-card">\n<div class="person-title">👤 {safe_person} (共 {len(p_list)} 張照片)</div>\n<div class="photo-grid">\n'
    for path_str in p_list[:12]:
        p_obj = Path(path_str)
        fname = html_lib.escape(p_obj.name)
        thumb_tag = ''
        try:
            with Image.open(p_obj) as im:
                im_rgb = ImageOps.exif_transpose(im).convert('RGB')
                im_rgb.thumbnail((320, 320))
                buf = io.BytesIO()
                im_rgb.save(buf, format='JPEG', quality=75)
                b64 = base64.b64encode(buf.getvalue()).decode('ascii')
                thumb_tag = f'<img src="data:image/jpeg;base64,{b64}" alt="{fname}" loading="lazy">'
        except Exception:
            pass
        html_content += f'<div class="photo-item">{thumb_tag}<div class="photo-caption">{fname}</div></div>\n'
    html_content += '</div>\n</div>\n'

html_content += '''
</body>
</html>
'''

album_html_file.write_text(html_content, encoding='utf-8')
print(f'✨ 已生成單檔離線相簿網頁：{album_html_file}')
print('日後在 Google Drive 下載或直接點開該 HTML 即可秒看人物畫廊！')

✨ 已生成單檔離線相簿網頁：/content/drive/MyDrive/Family_Album.html
日後在 Google Drive 下載或直接點開該 HTML 即可秒看人物畫廊！
